In [1]:
import os
import re
import glob
import json
from datetime import datetime
from fractions import Fraction
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne
from scipy.signal import resample_poly

In [2]:
# Settings for the block 1 + block 2 analysis.

# Point this to the parent folder that contains both block folders.
# raw_data_of_all_participants block 1
# raw_data_of_all_participants block 2
ROOT_RAW_DIR = r"C:\Users\Asus\Documents\Professor Francesca Starita"

# Search only inside the two block folders.
FILE_GLOB = os.path.join(ROOT_RAW_DIR, "raw_data_of_all_participants block *", "*.vhdr")

RUN_SCOPE_LABEL = "block1_block2"
ALLOWED_BLOCKS = {1, 2}

# Expected file count for a complete block 1 + block 2 run.
EXPECTED_N_VHDR = 64

# Set the main output folder.
OUT_DIR = r"D:\EEG_CleanSegments_And_Datasets"
os.makedirs(OUT_DIR, exist_ok=True)

# Create the output folder for the DC tail results.
DC_TAIL_OUT_DIR = os.path.join(OUT_DIR, "dc_tail_block1_block2")
os.makedirs(DC_TAIL_OUT_DIR, exist_ok=True)

# Config files used by the main preprocessing pipeline.
CONFIG_JSON_NAME = "DC_TAIL_S_recommended.json"
CONFIG_TXT_NAME = "DC_TAIL_S_recommended.txt"
CONFIG_JSON_PATH = os.path.join(OUT_DIR, CONFIG_JSON_NAME)
CONFIG_TXT_PATH = os.path.join(OUT_DIR, CONFIG_TXT_NAME)

# Match common DC marker labels in BrainVision annotations.
DC_PATTERN = re.compile(
    r"\bDC\b|DC\s*Correction|DCC|DC-corr|DC\s*correction/?",
    re.IGNORECASE
)

# This should match the main preprocessing pipeline.
BAND = (0.5, 100.0)

# Optional resampling target.
TARGET_SFREQ = 250.0

# Set the time window around each DC onset.
PRE_S = 5.0
POST_S = 12.0

# Set the baseline window before DC onset.
BASELINE_WIN = (-4.5, -0.5)

# Threshold based on the baseline RMS percentile.
BASELINE_PCTL = 95.0

# Recovery is the first time after DC when RMS stays below threshold.
# Require this to hold for at least STABLE_S seconds.
STABLE_S = 0.5

# Recommended DC_TAIL_S = p95(recovery_times) + safety margin.
SAFETY_MARGIN_S = 0.5

# Apply light smoothing to the RMS curve.
RMS_SMOOTH_S = 0.05

# Limit the number of curves used in the representative plot.
MAX_CURVES_FOR_REPRESENTATIVE = 500

# Warn if old snippet based outputs are still present.
snippet_like = sorted(glob.glob(os.path.join(OUT_DIR, "*first5min*")))
if len(snippet_like) > 0:
    print("[WARN] Found snippet-only outputs containing 'first5min'.")
    print("They should not be used as final thesis evidence.")
    for p in snippet_like[:10]:
        print(" -", p)

# Check how many VHDR files were found across block 1 and block 2.
found_vhdr = sorted(glob.glob(FILE_GLOB))
print(f"Found VHDR files across block 1 + block 2: {len(found_vhdr)}")

if len(found_vhdr) != EXPECTED_N_VHDR:
    print(f"[NOTE] Expected about {EXPECTED_N_VHDR} VHDR files for block 1 + block 2, found {len(found_vhdr)}.")

print("Run scope:", RUN_SCOPE_LABEL)
print("Allowed blocks:", sorted(list(ALLOWED_BLOCKS)))
print("Example files:")
for p in found_vhdr[:5]:
    print(" -", p)

Found VHDR files across block 1 + block 2: 60
[NOTE] Expected about 64 VHDR files for block 1 + block 2, found 60.
Run scope: block1_block2
Allowed blocks: [1, 2]
Example files:
 - C:\Users\Asus\Documents\Professor Francesca Starita\raw_data_of_all_participants block 1\01_ln_block1.vhdr
 - C:\Users\Asus\Documents\Professor Francesca Starita\raw_data_of_all_participants block 1\02gd_block1.vhdr
 - C:\Users\Asus\Documents\Professor Francesca Starita\raw_data_of_all_participants block 1\03as_block1.vhdr
 - C:\Users\Asus\Documents\Professor Francesca Starita\raw_data_of_all_participants block 1\04ep_block1.vhdr
 - C:\Users\Asus\Documents\Professor Francesca Starita\raw_data_of_all_participants block 1\05br_block1.vhdr


In [3]:
def parse_subject_block(path: str):
    """
    Example:
    02gd_block1.vhdr -> subject_id='02gd', block_id=1
    """
    base = os.path.basename(path)
    m = re.match(r"(.+?)_block(\d+)\.vhdr$", base, flags=re.IGNORECASE)
    if m:
        return m.group(1), int(m.group(2))
    return os.path.splitext(base)[0], None


def _read_vhdr_refs(vhdr_path: str):
    data_ref, marker_ref = None, None
    with open(vhdr_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            low = line.strip().lower()
            if low.startswith("datafile="):
                data_ref = line.split("=", 1)[1].strip().strip('"')
            elif low.startswith("markerfile="):
                marker_ref = line.split("=", 1)[1].strip().strip('"')
    return data_ref, marker_ref


def _try_candidates(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None


def _resolve_sidecar(vhdr_path: str, ref: str, exts):
    vhdr_dir = os.path.dirname(vhdr_path)
    stem = os.path.splitext(os.path.basename(vhdr_path))[0]

    candidates = []

    if ref:
        if os.path.isabs(ref):
            candidates.extend([
                ref,
                os.path.join(vhdr_dir, os.path.basename(ref)),
            ])
        else:
            candidates.extend([
                os.path.join(vhdr_dir, ref),
                os.path.join(vhdr_dir, os.path.basename(ref)),
            ])

    for ext in exts:
        candidates.append(os.path.join(vhdr_dir, stem + ext))

    found = _try_candidates(candidates)
    if found is not None:
        return found

    try:
        for fn in os.listdir(vhdr_dir):
            if fn.lower().startswith(stem.lower()) and any(fn.lower().endswith(ext.lower()) for ext in exts):
                p = os.path.join(vhdr_dir, fn)
                if os.path.exists(p):
                    return p
    except Exception:
        pass

    return None


def _safe_read_raw_brainvision(vhdr_path: str) -> mne.io.BaseRaw:
    try:
        return mne.io.read_raw_brainvision(vhdr_path, preload=False, verbose="ERROR")
    except Exception:
        data_ref, marker_ref = _read_vhdr_refs(vhdr_path)
        data_path = _resolve_sidecar(vhdr_path, data_ref, exts=[".eeg", ".dat"])
        marker_path = _resolve_sidecar(vhdr_path, marker_ref, exts=[".vmrk"])

        if data_path is None or marker_path is None:
            raise RuntimeError(f"Could not resolve BrainVision sidecars for: {vhdr_path}")

        fixed_dir = os.path.join(OUT_DIR, "_vhdr_fixed_for_dc_tail")
        os.makedirs(fixed_dir, exist_ok=True)
        fixed_vhdr = os.path.join(fixed_dir, os.path.basename(vhdr_path))

        fixed_lines = []
        with open(vhdr_path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                low = line.strip().lower()
                if low.startswith("datafile="):
                    fixed_lines.append(f"DataFile={data_path}\n")
                elif low.startswith("markerfile="):
                    fixed_lines.append(f"MarkerFile={marker_path}\n")
                else:
                    fixed_lines.append(line if line.endswith("\n") else line + "\n")

        with open(fixed_vhdr, "w", encoding="utf-8", errors="ignore") as f:
            f.writelines(fixed_lines)

        return mne.io.read_raw_brainvision(fixed_vhdr, preload=False, verbose="ERROR")


def read_raw_any(path: str) -> mne.io.BaseRaw:
    """
    Load a BrainVision or FIF file and keep EEG channels only.
    """
    if path.lower().endswith(".vhdr"):
        raw = _safe_read_raw_brainvision(path)
    elif path.lower().endswith(".fif"):
        raw = mne.io.read_raw_fif(path, preload=False, verbose="ERROR")
    else:
        raise ValueError(f"Unsupported file type: {path}")

    picks = mne.pick_types(raw.info, eeg=True, eog=False, stim=False, misc=False, exclude=[])

    # Fallback for files with unusual channel typing.
    if len(picks) == 0:
        picks = mne.pick_types(raw.info, eeg=True, eog=False, stim=False, misc=True, exclude=[])

    if len(picks) == 0:
        raise RuntimeError("No EEG/misc channels available after picking. Check raw.info channel types.")

    raw.pick(picks)
    return raw


def build_annotation_table(raw: mne.io.BaseRaw) -> pd.DataFrame:
    """
    Return a sorted annotation table with onset, description, and marker gap.
    """
    if len(raw.annotations) == 0:
        return pd.DataFrame(columns=["onset_s", "desc", "next_onset_s", "gap_to_next_s"])

    ann_df = pd.DataFrame({
        "onset_s": np.asarray(raw.annotations.onset, dtype=float),
        "desc": [str(d) for d in raw.annotations.description],
    }).sort_values("onset_s").reset_index(drop=True)

    ann_df["next_onset_s"] = ann_df["onset_s"].shift(-1)
    ann_df["gap_to_next_s"] = ann_df["next_onset_s"] - ann_df["onset_s"]
    return ann_df


def get_dc_events(raw: mne.io.BaseRaw) -> pd.DataFrame:
    """
    Return only the DC marker rows.
    """
    ann_df = build_annotation_table(raw)
    if len(ann_df) == 0:
        return ann_df

    dc_mask = ann_df["desc"].apply(lambda x: bool(DC_PATTERN.search(str(x))))
    dc_df = ann_df.loc[dc_mask, ["onset_s", "desc", "gap_to_next_s"]].copy().reset_index(drop=True)
    return dc_df


def robust_resample(data: np.ndarray, sf_in: float, sf_out: float):
    """
    Resample with a rational approximation.
    """
    if sf_out is None or abs(sf_in - sf_out) < 1e-12:
        return data, sf_in

    ratio = sf_out / sf_in
    frac = Fraction(ratio).limit_denominator(1000)
    up, down = frac.numerator, frac.denominator

    data_rs = resample_poly(data, up=up, down=down, axis=-1)
    sf_eff = sf_in * up / down
    return data_rs, sf_eff

In [4]:
def moving_average(x: np.ndarray, win_samp: int):
    x = np.asarray(x, dtype=float)
    if win_samp <= 1 or x.size == 0:
        return x
    kernel = np.ones(int(win_samp), dtype=float) / float(win_samp)
    return np.convolve(x, kernel, mode="same")


def estimate_dc_recovery_one_event(
    raw: mne.io.BaseRaw,
    dc_t: float,
    band=BAND,
    pre_s=PRE_S,
    post_s=POST_S,
    baseline_win=BASELINE_WIN,
    baseline_pctl=BASELINE_PCTL,
    target_sfreq=TARGET_SFREQ,
    stable_s=STABLE_S,
    rms_smooth_s=RMS_SMOOTH_S,
):
    """
    Estimate DC recovery time from the signal.

    Returns a dict with:
    - status
    - recovery_time_s
    - baseline_thr
    - sf_used
    - time_rel
    - rms_smooth
    """

    sf = float(raw.info["sfreq"])

    # Use a full pre and post window for each event.
    tmin = float(dc_t) - float(pre_s)
    tmax = float(dc_t) + float(post_s)

    if tmin < 0.0:
        return {
            "status": "skip_insufficient_pre_window",
            "recovery_time_s": np.nan,
            "baseline_thr": np.nan,
            "sf_used": np.nan,
            "time_rel": None,
            "rms_smooth": None,
        }

    if tmax > float(raw.times[-1]):
        return {
            "status": "skip_insufficient_post_window",
            "recovery_time_s": np.nan,
            "baseline_thr": np.nan,
            "sf_used": np.nan,
            "time_rel": None,
            "rms_smooth": None,
        }

    s0 = int(round(tmin * sf))
    s1 = int(round(tmax * sf))

    if s1 <= s0:
        return {
            "status": "skip_invalid_window",
            "recovery_time_s": np.nan,
            "baseline_thr": np.nan,
            "sf_used": np.nan,
            "time_rel": None,
            "rms_smooth": None,
        }

    data = raw.get_data(start=s0, stop=s1)
    if data.size == 0 or data.shape[-1] < 2:
        return {
            "status": "skip_empty_data",
            "recovery_time_s": np.nan,
            "baseline_thr": np.nan,
            "sf_used": np.nan,
            "time_rel": None,
            "rms_smooth": None,
        }

    # Match the main preprocessing band, but keep it below Nyquist.
    l_freq, h_freq = float(band[0]), float(band[1])
    nyq = sf / 2.0
    h_freq = min(h_freq, nyq * 0.99)

    if h_freq <= l_freq:
        return {
            "status": "skip_invalid_band",
            "recovery_time_s": np.nan,
            "baseline_thr": np.nan,
            "sf_used": np.nan,
            "time_rel": None,
            "rms_smooth": None,
        }

    try:
        data_f = mne.filter.filter_data(
            data,
            sfreq=sf,
            l_freq=l_freq,
            h_freq=h_freq,
            verbose="ERROR"
        )
    except Exception:
        return {
            "status": "skip_filter_error",
            "recovery_time_s": np.nan,
            "baseline_thr": np.nan,
            "sf_used": np.nan,
            "time_rel": None,
            "rms_smooth": None,
        }

    # Resample if needed
    try:
        data_rs, sf_used = robust_resample(data_f, sf_in=sf, sf_out=target_sfreq)
    except Exception:
        return {
            "status": "skip_resample_error",
            "recovery_time_s": np.nan,
            "baseline_thr": np.nan,
            "sf_used": np.nan,
            "time_rel": None,
            "rms_smooth": None,
        }

    if data_rs.size == 0 or data_rs.shape[-1] < 2:
        return {
            "status": "skip_empty_after_resample",
            "recovery_time_s": np.nan,
            "baseline_thr": np.nan,
            "sf_used": np.nan,
            "time_rel": None,
            "rms_smooth": None,
        }

    # RMS across channels
    rms = np.sqrt(np.mean(np.square(data_rs), axis=0))

    # Light smoothing for a cleaner recovery curve.
    smooth_win = max(1, int(round(float(rms_smooth_s) * float(sf_used))))
    rms_s = moving_average(rms, smooth_win)

    # Time relative to DC onset
    t_rel = (np.arange(rms_s.size, dtype=float) / float(sf_used)) - float(pre_s)

    # Baseline threshold
    b0, b1 = float(baseline_win[0]), float(baseline_win[1])
    base_mask = (t_rel >= b0) & (t_rel <= b1)

    min_baseline_samples = max(10, int(round(0.2 * float(sf_used))))
    if int(base_mask.sum()) < min_baseline_samples:
        return {
            "status": "skip_missing_baseline",
            "recovery_time_s": np.nan,
            "baseline_thr": np.nan,
            "sf_used": float(sf_used),
            "time_rel": t_rel,
            "rms_smooth": rms_s,
        }

    thr = float(np.percentile(rms_s[base_mask], float(baseline_pctl)))

    # Recovery starts at the first time after DC onset when RMS stays.
    # Below the threshold for at least stable_s seconds.
    stable_n = max(1, int(round(float(stable_s) * float(sf_used))))
    below = (rms_s <= thr).astype(np.int8)
    conv = np.convolve(below, np.ones(stable_n, dtype=np.int32), mode="valid")

    candidate_idx = np.where((conv == stable_n) & (t_rel[:conv.size] >= 0.0))[0]

    if candidate_idx.size == 0:
        return {
            "status": "no_recovery_within_window",
            "recovery_time_s": np.nan,
            "baseline_thr": thr,
            "sf_used": float(sf_used),
            "time_rel": t_rel,
            "rms_smooth": rms_s,
        }

    rec_t = float(t_rel[int(candidate_idx[0])])

    return {
        "status": "ok",
        "recovery_time_s": rec_t,
        "baseline_thr": thr,
        "sf_used": float(sf_used),
        "time_rel": t_rel,
        "rms_smooth": rms_s,
    }

In [5]:
files = sorted(glob.glob(FILE_GLOB))
files = [f for f in files if parse_subject_block(f)[1] in ALLOWED_BLOCKS]

print("Found VHDR files across block 1 + block 2:", len(files))
if len(files) == 0:
    raise FileNotFoundError(
        "No .vhdr files found. Check ROOT_RAW_DIR and FILE_GLOB for the block 1 + block 2 setup."
    )

print("First files:")
for f in files[:5]:
    print(" -", f)

Found VHDR files across block 1 + block 2: 60
First files:
 - C:\Users\Asus\Documents\Professor Francesca Starita\raw_data_of_all_participants block 1\01_ln_block1.vhdr
 - C:\Users\Asus\Documents\Professor Francesca Starita\raw_data_of_all_participants block 1\02gd_block1.vhdr
 - C:\Users\Asus\Documents\Professor Francesca Starita\raw_data_of_all_participants block 1\03as_block1.vhdr
 - C:\Users\Asus\Documents\Professor Francesca Starita\raw_data_of_all_participants block 1\04ep_block1.vhdr
 - C:\Users\Asus\Documents\Professor Francesca Starita\raw_data_of_all_participants block 1\05br_block1.vhdr


In [6]:
# Scan all block 1 + block 2 files and estimate signal based DC recovery.

rows = []
failed_files = []
curve_bank = []
curve_thr_bank = []

files_processed = 0
files_with_dc = 0
files_without_dc = 0

subjects_seen = set()
blocks_seen = set()
n_dc_markers_total = 0

for f in files:
    try:
        subj, block = parse_subject_block(f)
        raw = read_raw_any(f)

        files_processed += 1
        subjects_seen.add(subj)
        blocks_seen.add(block)

        dc_df = get_dc_events(raw)

        # Some files may not contain any DC markers.
        if len(dc_df) == 0:
            files_without_dc += 1
            del raw
            continue

        files_with_dc += 1
        n_dc_markers_total += int(len(dc_df))

        for _, ev in dc_df.iterrows():
            dc_t = float(ev["onset_s"])
            dc_desc = str(ev["desc"])
            dc_gap = float(ev["gap_to_next_s"]) if pd.notna(ev["gap_to_next_s"]) else np.nan

            est = estimate_dc_recovery_one_event(
                raw=raw,
                dc_t=dc_t,
                band=BAND,
                pre_s=PRE_S,
                post_s=POST_S,
                baseline_win=BASELINE_WIN,
                baseline_pctl=BASELINE_PCTL,
                target_sfreq=TARGET_SFREQ,
                stable_s=STABLE_S,
                rms_smooth_s=RMS_SMOOTH_S,
            )

            rows.append({
                "subject_id": str(subj),
                "block_id": int(block) if block is not None else np.nan,
                "file_path": str(f),
                "file_name": os.path.basename(f),

                "dc_onset_s": float(dc_t),
                "dc_marker_desc": dc_desc,
                "dc_marker_gap_s": dc_gap,

                "band_lo": float(BAND[0]),
                "band_hi": float(BAND[1]),
                "target_sfreq": float(TARGET_SFREQ) if TARGET_SFREQ is not None else np.nan,
                "sf_used": float(est["sf_used"]) if np.isfinite(est["sf_used"]) else np.nan,

                "baseline_win": f"{BASELINE_WIN[0]},{BASELINE_WIN[1]}",
                "baseline_pctl": float(BASELINE_PCTL),
                "baseline_thr95": float(est["baseline_thr"]) if np.isfinite(est["baseline_thr"]) else np.nan,

                "pre_s": float(PRE_S),
                "post_s": float(POST_S),
                "stable_s": float(STABLE_S),
                "rms_smooth_s": float(RMS_SMOOTH_S),

                "recovery_time_s": float(est["recovery_time_s"]) if np.isfinite(est["recovery_time_s"]) else np.nan,
                "status": str(est["status"]),
            })

            if est["status"] == "ok" and est["time_rel"] is not None and est["rms_smooth"] is not None:
                if len(curve_bank) < MAX_CURVES_FOR_REPRESENTATIVE:
                    curve_bank.append(est["rms_smooth"].astype(np.float32, copy=False))
                    curve_thr_bank.append(float(est["baseline_thr"]))

        del raw

    except Exception as e:
        failed_files.append({
            "file_path": str(f),
            "file_name": os.path.basename(f),
            "error": str(e),
        })

df_dc = pd.DataFrame(rows)

# Sort the table for easier review
if len(df_dc) > 0:
    df_dc = df_dc.sort_values(
        ["subject_id", "block_id", "file_name", "dc_onset_s"],
        ascending=[True, True, True, True]
    ).reset_index(drop=True)

out_csv = os.path.join(DC_TAIL_OUT_DIR, "dc_tail_recovery_table_block1_block2.csv")
df_dc.to_csv(out_csv, index=False)

failed_csv = os.path.join(DC_TAIL_OUT_DIR, "dc_tail_failed_files_block1_block2.csv")
pd.DataFrame(
    failed_files,
    columns=["file_path", "file_name", "error"]
).to_csv(failed_csv, index=False)

# Save a short scan summary.
scan_summary = {
    "n_vhdr_found": int(len(files)),
    "n_files_processed": int(files_processed),
    "n_files_with_dc": int(files_with_dc),
    "n_files_without_dc": int(files_without_dc),
    "n_failed_files": int(len(failed_files)),
    "n_subjects_seen": int(len(subjects_seen)),
    "blocks_seen": sorted([int(b) for b in blocks_seen if b is not None]),
    "n_dc_markers_total": int(n_dc_markers_total),
    "n_rows_in_dc_table": int(len(df_dc)),
}
scan_summary_json = os.path.join(DC_TAIL_OUT_DIR, "dc_tail_scan_summary_block1_block2.json")
with open(scan_summary_json, "w", encoding="utf-8") as f:
    json.dump(scan_summary, f, indent=2)

print("Saved:", out_csv)
print("Saved:", failed_csv)
print("Saved:", scan_summary_json)

print("\n=== BLOCK-1 + BLOCK-2 DC SCAN SUMMARY ===")
print("Rows in df_dc:", len(df_dc))
print("Files found:", len(files))
print("Files processed:", files_processed)
print("Files with DC:", files_with_dc)
print("Files without DC:", files_without_dc)
print("Files failed:", len(failed_files))
print("Unique subjects:", len(subjects_seen))
print("Blocks seen:", sorted([b for b in blocks_seen if b is not None]))
print("Total DC markers:", n_dc_markers_total)

if len(df_dc) > 0:
    print("\nStatus counts:")
    print(df_dc["status"].value_counts(dropna=False))

df_dc.head()

Saved: D:\EEG_CleanSegments_And_Datasets\dc_tail_block1_block2\dc_tail_recovery_table_block1_block2.csv
Saved: D:\EEG_CleanSegments_And_Datasets\dc_tail_block1_block2\dc_tail_failed_files_block1_block2.csv
Saved: D:\EEG_CleanSegments_And_Datasets\dc_tail_block1_block2\dc_tail_scan_summary_block1_block2.json

=== BLOCK-1 + BLOCK-2 DC SCAN SUMMARY ===
Rows in df_dc: 310
Files found: 60
Files processed: 60
Files with DC: 59
Files without DC: 1
Files failed: 0
Unique subjects: 31
Blocks seen: [1, 2]
Total DC markers: 310

Status counts:
status
ok                               273
skip_insufficient_pre_window      36
skip_insufficient_post_window      1
Name: count, dtype: int64


,subject_id,block_id,file_path,file_name,dc_onset_s,dc_marker_desc,dc_marker_gap_s,band_lo,band_hi,target_sfreq,sf_used,baseline_win,baseline_pctl,baseline_thr95,pre_s,post_s,stable_s,rms_smooth_s,recovery_time_s,status
0,01_ln,1,C:\Users\Asus\Documents\Professor Francesca St...,01_ln_block1.vhdr,3.66,DC Correction/,17.2450,0.5,100.0,250.0,NaN,"-4.5,-0.5",95.0,NaN,5.0,12.0,0.5,0.05,NaN,skip_insufficient_pre_window
1,01_ln,1,C:\Users\Asus\Documents\Professor Francesca St...,01_ln_block1.vhdr,130.14,DC Correction/,7.5840,0.5,100.0,250.0,250.0,"-4.5,-0.5",95.0,0.000150,5.0,12.0,0.5,0.05,0.900,ok
2,01_ln,1,C:\Users\Asus\Documents\Professor Francesca St...,01_ln_block1.vhdr,634.36,DC Correction/,6.8798,0.5,100.0,250.0,250.0,"-4.5,-0.5",95.0,0.000199,5.0,12.0,0.5,0.05,0.644,ok
3,01_ln,2,C:\Users\Asus\Documents\Professor Francesca St...,01_ln_block2.vhdr,2.52,DC Correction/,17.0340,0.5,100.0,250.0,NaN,"-4.5,-0.5",95.0,NaN,5.0,12.0,0.5,0.05,NaN,skip_insufficient_pre_window
4,01_ln,2,C:\Users\Asus\Documents\Professor Francesca St...,01_ln_block2.vhdr,88.38,DC Correction/,4.3344,0.5,100.0,250.0,250.0,"-4.5,-0.5",95.0,0.000138,5.0,12.0,0.5,0.05,1.020,ok


In [7]:
# Summarize the block 1 + block 2 recovery distribution and save the figures.

valid_df = df_dc.loc[
    (df_dc["status"] == "ok") &
    (df_dc["recovery_time_s"].notna())
].copy()

valid = valid_df["recovery_time_s"].values.astype(float)

print("N total DC markers found:", int(n_dc_markers_total))
print("N DC events with valid recovery:", int(valid.size))

if valid.size == 0:
    print("\nNo valid recovery values found.")
    print("Possible reasons:")
    print("  1) DC_PATTERN does not match the real annotation names.")
    print("  2) Many DC events do not have enough pre/post window.")
    print("  3) The recovery criterion is too strict for the current window.")
    print("\nRun the debug block below and inspect the annotation names.")
else:
    p05 = float(np.percentile(valid, 5))
    p25 = float(np.percentile(valid, 25))
    p50 = float(np.percentile(valid, 50))
    p75 = float(np.percentile(valid, 75))
    p90 = float(np.percentile(valid, 90))
    p95 = float(np.percentile(valid, 95))
    p99 = float(np.percentile(valid, 99))
    mean_v = float(np.mean(valid))
    std_v = float(np.std(valid))
    min_v = float(np.min(valid))
    max_v = float(np.max(valid))

    recommended = float(p95 + SAFETY_MARGIN_S)

    print(f"min recovery : {min_v:.3f} s")
    print(f"p05 recovery : {p05:.3f} s")
    print(f"p25 recovery : {p25:.3f} s")
    print(f"p50 recovery : {p50:.3f} s")
    print(f"p75 recovery : {p75:.3f} s")
    print(f"p90 recovery : {p90:.3f} s")
    print(f"p95 recovery : {p95:.3f} s")
    print(f"p99 recovery : {p99:.3f} s")
    print(f"max recovery : {max_v:.3f} s")
    print(f"mean ± std   : {mean_v:.3f} ± {std_v:.3f} s")
    print(f"Recommended DC_TAIL_S = p95 + margin({SAFETY_MARGIN_S:.2f}) = {recommended:.3f} s")

    # Save the valid event table.
    valid_csv = os.path.join(DC_TAIL_OUT_DIR, "dc_recovery_valid_events_block1_block2.csv")
    valid_df.to_csv(valid_csv, index=False)
    print("Saved:", valid_csv)

    # Save the summary table.
    pct_df = pd.DataFrame([{
        "n_dc_events_valid": int(valid.size),
        "min_recovery_s": min_v,
        "p05_recovery_s": p05,
        "p25_recovery_s": p25,
        "p50_recovery_s": p50,
        "p75_recovery_s": p75,
        "p90_recovery_s": p90,
        "p95_recovery_s": p95,
        "p99_recovery_s": p99,
        "max_recovery_s": max_v,
        "mean_recovery_s": mean_v,
        "std_recovery_s": std_v,
        "recommended_dc_tail_s": recommended,
        "safety_margin_s": float(SAFETY_MARGIN_S),
    }])
    pct_csv = os.path.join(DC_TAIL_OUT_DIR, "dc_recovery_time_percentiles_block1_block2.csv")
    pct_df.to_csv(pct_csv, index=False)
    print("Saved:", pct_csv)

    # Save the histogram.
    hist_png = os.path.join(DC_TAIL_OUT_DIR, "dc_recovery_time_hist_block1_block2.png")
    plt.figure(figsize=(7, 5))
    plt.hist(valid, bins=30)
    plt.axvline(p95, linestyle="--", label=f"p95 = {p95:.2f}s")
    plt.axvline(recommended, linestyle="--", label=f"p95 + margin = {recommended:.2f}s")
    plt.xlabel("DC recovery time (s)")
    plt.ylabel("Count")
    plt.title("DC recovery time distribution (block 1 + block 2)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(hist_png, dpi=180)
    plt.close()
    print("Saved:", hist_png)

    # Save the boxplot.
    box_png = os.path.join(DC_TAIL_OUT_DIR, "dc_recovery_time_boxplot_block1_block2.png")
    plt.figure(figsize=(5, 4.5))
    plt.boxplot(valid, vert=True)
    plt.ylabel("DC recovery time (s)")
    plt.title("DC recovery time summary (block 1 + block 2)")
    plt.tight_layout()
    plt.savefig(box_png, dpi=180)
    plt.close()
    print("Saved:", box_png)

    # Compare marker gap and signal based recovery.
    gap_scatter_png = os.path.join(DC_TAIL_OUT_DIR, "dc_marker_gap_vs_recovery_block1_block2.png")
    gap_df = valid_df.loc[valid_df["dc_marker_gap_s"].notna()].copy()

    if len(gap_df) > 0:
        plt.figure(figsize=(6.5, 5))
        plt.scatter(
            gap_df["dc_marker_gap_s"].values,
            gap_df["recovery_time_s"].values,
            alpha=0.7
        )
        lim_max = float(max(
            gap_df["dc_marker_gap_s"].max(),
            gap_df["recovery_time_s"].max()
        ))
        plt.plot([0, lim_max], [0, lim_max], linestyle="--", label="y = x")
        plt.xlabel("DC marker-gap duration (s)")
        plt.ylabel("Signal-based recovery time (s)")
        plt.title("Marker-gap duration vs signal-based recovery (block 1 + block 2)")
        plt.legend()
        plt.tight_layout()
        plt.savefig(gap_scatter_png, dpi=180)
        plt.close()
        print("Saved:", gap_scatter_png)

    # Save the representative median RMS curve.
    if len(curve_bank) > 0:
        min_len = min(len(x) for x in curve_bank)
        curve_stack = np.vstack([x[:min_len] for x in curve_bank])
        median_rms = np.median(curve_stack, axis=0)
        q25_rms = np.percentile(curve_stack, 25, axis=0)
        q75_rms = np.percentile(curve_stack, 75, axis=0)
        thr_rep = float(np.median(curve_thr_bank))

        sf_for_plot = float(TARGET_SFREQ) if TARGET_SFREQ is not None else float(valid_df["sf_used"].dropna().iloc[0])
        t_rel = (np.arange(min_len) / sf_for_plot) - float(PRE_S)

        rep_png = os.path.join(DC_TAIL_OUT_DIR, "dc_tail_median_rms_block1_block2.png")
        plt.figure(figsize=(8, 4.5))
        plt.plot(t_rel, median_rms, label="Median RMS across DC events")
        plt.fill_between(t_rel, q25_rms, q75_rms, alpha=0.25, label="IQR")
        plt.axhline(thr_rep, linestyle="--", label=f"Median baseline p{int(BASELINE_PCTL)} threshold")
        plt.axvline(0.0, linestyle="--", label="DC onset")
        plt.axvline(recommended, linestyle="--", label=f"Recommended DC_TAIL_S = {recommended:.2f}s")
        plt.xlabel("Time relative to DC onset (s)")
        plt.ylabel("RMS (a.u.)")
        plt.title("Representative DC recovery curve (block 1 + block 2)")
        plt.legend()
        plt.tight_layout()
        plt.savefig(rep_png, dpi=180)
        plt.close()
        print("Saved:", rep_png)

    print("\nInterpretation:")
    print(" - dc_marker_gap_s is the annotation-based time to the next marker.")
    print(" - recovery_time_s is signal-based and shows how long the DC effect remains in the EEG.")
    print(" - DC_TAIL_S should be chosen from the recovery_time_s distribution, not from marker-gap alone.")
    print(" - In this notebook, the recommended value is based on p95(recovery_time_s) + safety margin for block 1 + block 2.")

N total DC markers found: 310
N DC events with valid recovery: 273
min recovery : 0.456 s
p05 recovery : 0.624 s
p25 recovery : 0.676 s
p50 recovery : 0.992 s
p75 recovery : 1.124 s
p90 recovery : 1.216 s
p95 recovery : 1.336 s
p99 recovery : 1.581 s
max recovery : 1.884 s
mean ± std   : 0.949 ± 0.247 s
Recommended DC_TAIL_S = p95 + margin(0.50) = 1.836 s
Saved: D:\EEG_CleanSegments_And_Datasets\dc_tail_block1_block2\dc_recovery_valid_events_block1_block2.csv
Saved: D:\EEG_CleanSegments_And_Datasets\dc_tail_block1_block2\dc_recovery_time_percentiles_block1_block2.csv
Saved: D:\EEG_CleanSegments_And_Datasets\dc_tail_block1_block2\dc_recovery_time_hist_block1_block2.png
Saved: D:\EEG_CleanSegments_And_Datasets\dc_tail_block1_block2\dc_recovery_time_boxplot_block1_block2.png
Saved: D:\EEG_CleanSegments_And_Datasets\dc_tail_block1_block2\dc_marker_gap_vs_recovery_block1_block2.png
Saved: D:\EEG_CleanSegments_And_Datasets\dc_tail_block1_block2\dc_tail_median_rms_block1_block2.png

Interpret

In [8]:
from datetime import datetime

valid_df = df_dc.loc[
    (df_dc["status"] == "ok") &
    (df_dc["recovery_time_s"].notna())
].copy()

valid = valid_df["recovery_time_s"].values.astype(float)

if valid.size == 0:
    print("No valid recovery times -> cannot recommend DC_TAIL_S yet.")
else:
    p50 = float(np.percentile(valid, 50))
    p75 = float(np.percentile(valid, 75))
    p90 = float(np.percentile(valid, 90))
    p95 = float(np.percentile(valid, 95))
    p99 = float(np.percentile(valid, 99))
    recommended = float(p95 + SAFETY_MARGIN_S)

    # Save the final value in a plain text file for quick reuse.
    with open(CONFIG_TXT_PATH, "w", encoding="utf-8") as f:
        f.write(f"{recommended:.6f}\n")

    # Save the same recommendation together with the analysis metadata.
    payload = {
        "schema_version": 2,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "analysis_type": "signal_based_dc_recovery_block1_block2",

        "root_raw_dir": ROOT_RAW_DIR,
        "file_glob": FILE_GLOB,
        "dc_pattern": DC_PATTERN.pattern,

        "DC_TAIL_S": recommended,
        "p50_recovery_s": p50,
        "p75_recovery_s": p75,
        "p90_recovery_s": p90,
        "p95_recovery_s": p95,
        "p99_recovery_s": p99,
        "safety_margin_s": float(SAFETY_MARGIN_S),

        "band": {
            "l_freq": float(BAND[0]),
            "h_freq": float(BAND[1]),
        },
        "target_sfreq": None if TARGET_SFREQ is None else float(TARGET_SFREQ),
        "pre_s": float(PRE_S),
        "post_s": float(POST_S),
        "baseline_win": {
            "tmin": float(BASELINE_WIN[0]),
            "tmax": float(BASELINE_WIN[1]),
        },
        "baseline_percentile": float(BASELINE_PCTL),
        "stable_s": float(STABLE_S),
        "rms_smooth_s": float(RMS_SMOOTH_S),

        "n_vhdr_found": int(len(files)),
        "n_files_processed": int(files_processed),
        "n_failed_files": int(len(failed_files)),
        "n_subjects": int(len(subjects_seen)),
        "blocks_seen": sorted([int(b) for b in blocks_seen if b is not None]),
        "n_dc_markers_total": int(n_dc_markers_total),
        "n_dc_events_valid": int(valid.size),

        "notes": [
            "This analysis was run on block 1 + block 2.",
            "Marker-gap duration is annotation-based only.",
            "DC_TAIL_S is justified from signal-based RMS recovery.",
            "Recommended value = p95(recovery_time_s) + safety margin."
        ]
    }

    with open(CONFIG_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    # Keep one copy inside the DC tail results folder as well.
    json_copy_path = os.path.join(DC_TAIL_OUT_DIR, CONFIG_JSON_NAME)
    txt_copy_path = os.path.join(DC_TAIL_OUT_DIR, CONFIG_TXT_NAME)

    with open(json_copy_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    with open(txt_copy_path, "w", encoding="utf-8") as f:
        f.write(f"{recommended:.6f}\n")

    print("Recommended DC_TAIL_S =", f"{recommended:.3f}", "s")
    print("Saved:", CONFIG_TXT_PATH)
    print("Saved:", CONFIG_JSON_PATH)
    print("Saved copy:", txt_copy_path)
    print("Saved copy:", json_copy_path)

    print("\nBLOCK-1 + BLOCK-2 verification summary:")
    print(" - VHDR found:", len(files))
    print(" - Files processed:", files_processed)
    print(" - Subjects:", len(subjects_seen))
    print(" - Blocks:", sorted([b for b in blocks_seen if b is not None]))
    print(" - DC markers total:", n_dc_markers_total)
    print(" - Valid DC recovery events:", valid.size)
    print(" - Failed files:", len(failed_files))

Recommended DC_TAIL_S = 1.836 s
Saved: D:\EEG_CleanSegments_And_Datasets\DC_TAIL_S_recommended.txt
Saved: D:\EEG_CleanSegments_And_Datasets\DC_TAIL_S_recommended.json
Saved copy: D:\EEG_CleanSegments_And_Datasets\dc_tail_block1_block2\DC_TAIL_S_recommended.txt
Saved copy: D:\EEG_CleanSegments_And_Datasets\dc_tail_block1_block2\DC_TAIL_S_recommended.json

BLOCK-1 + BLOCK-2 verification summary:
 - VHDR found: 60
 - Files processed: 60
 - Subjects: 31
 - Blocks: [1, 2]
 - DC markers total: 310
 - Valid DC recovery events: 273
 - Failed files: 0


In [9]:
# Check a few files and print their annotation labels.

sample_files = files[: min(5, len(files))]

if len(sample_files) == 0:
    print("No files available.")
else:
    for f in sample_files:
        try:
            raw0 = read_raw_any(f)

            if len(raw0.annotations) == 0:
                print()
                print("Example file:", os.path.basename(f))
                print("No annotations found.")
                continue

            descs = [str(d) for d in raw0.annotations.description]
            uniq = sorted(set(descs))

            print()
            print("Example file:", os.path.basename(f))
            print("Total annotations:", len(descs))
            print("Unique annotation descriptions:", len(uniq))
            print("Unique annotation descriptions (first 100):")
            for u in uniq[:100]:
                print(u)

        except Exception as e:
            print()
            print("Debug read failed:", os.path.basename(f))
            print(str(e))


Example file: 01_ln_block1.vhdr
Total annotations: 124
Unique annotation descriptions: 10
Unique annotation descriptions (first 100):
DC Correction/
New Segment/
Stimulus/S 11
Stimulus/S 12
Stimulus/S 13
Stimulus/S110
Stimulus/S114
Stimulus/S120
Stimulus/S124
Stimulus/S134

Example file: 02gd_block1.vhdr
Total annotations: 126
Unique annotation descriptions: 10
Unique annotation descriptions (first 100):
DC Correction/
New Segment/
Stimulus/S 11
Stimulus/S 12
Stimulus/S 13
Stimulus/S110
Stimulus/S114
Stimulus/S120
Stimulus/S124
Stimulus/S134

Example file: 03as_block1.vhdr
Total annotations: 147
Unique annotation descriptions: 10
Unique annotation descriptions (first 100):
DC Correction/
New Segment/
Stimulus/S 11
Stimulus/S 12
Stimulus/S 13
Stimulus/S110
Stimulus/S114
Stimulus/S120
Stimulus/S124
Stimulus/S134

Example file: 04ep_block1.vhdr
Total annotations: 124
Unique annotation descriptions: 10
Unique annotation descriptions (first 100):
DC Correction/
New Segment/
Stimulus/S 11
S